In [26]:
# Cell 1: Imports
import pandas as pd
import numpy as np
import re

pd.set_option("display.max_columns", 200)
pd.set_option("display.width", 200)

In [27]:
player_stats = pd.read_csv("../data/scrape/amateur/player_stats.csv")
matches = pd.read_csv("../data/scrape/amateur/matches.csv")
players = pd.read_csv("../data/scrape/amateur/players.csv")

In [28]:
def standardize_columns(df):
    df = df.copy()
    df.columns = (
        df.columns.astype(str)
        .str.strip()
        .str.lower()
        .str.replace(r"[^a-z0-9]+", "_", regex=True)
        .str.strip("_")
    )
    return df

matches = standardize_columns(matches)
player_stats = standardize_columns(player_stats)
players = standardize_columns(players)

print("matches columns:", matches.columns.tolist())
print("player_stats columns:", player_stats.columns.tolist())
print("players columns:", players.columns.tolist())

matches columns: ['match_id', 'season', 'game_date', 'league', 'home_club_id', 'away_club_id', 'home_goals', 'away_goals']
player_stats columns: ['player_id', 'match_id', 'club_id', 'goals', 'assists', 'yellow', 'yellow_red', 'red', 'start_eleven', 'minutes', 'on_min', 'off_min', 'team_goals', 'team_conceded', 'rating']
players columns: ['player_id', 'player_name', 'nationality', 'date_of_birth', 'height', 'position']


In [29]:
def pick_col(df, candidates, required=True):
    cols = list(df.columns)

    for c in candidates:
        if c in cols:
            return c

    for c in candidates:
        for col in cols:
            if c in col:
                return col

    if required:
        raise KeyError(f"Keine passende Spalte gefunden. Kandidaten: {candidates}")
    return None

In [30]:
PS = {
    "player_id": pick_col(player_stats, ["player_id"]),
    "match_id": pick_col(player_stats, ["match_id"]),
    "club_id": pick_col(player_stats, ["club_id"]),
    "season": pick_col(player_stats, ["season"], required=False),
    "goals": pick_col(player_stats, ["goals"], required=False),
    "assists": pick_col(player_stats, ["assists"], required=False),
    "yellow_cards": pick_col(player_stats, ["yellow"], required=False),
    "red_cards": pick_col(player_stats, ["red"], required=False),
    "yellow_red_cards": pick_col(player_stats, ["yellow_red"], required=False),
    "started": pick_col(player_stats, ["start_eleven"], required=False),
    "minutes": pick_col(player_stats, ["minutes"], required=False),
    "rating": pick_col(player_stats, ["rating"], required=False),
    "team_goals": pick_col(player_stats, ["team_goals"], required=False),
    "team_conceded": pick_col(player_stats, ["team_conceded"], required=False),
}

M = {
    "match_id": pick_col(matches, ["match_id"]),
    "season": pick_col(matches, ["season"], required=False),
    "competition": pick_col(matches, ["league"], required=False),
    "home_club_id": pick_col(matches, ["home_club_id"]),
    "away_club_id": pick_col(matches, ["away_club_id"]),
    "home_goals": pick_col(matches, ["home_goals"]),
    "away_goals": pick_col(matches, ["away_goals"]),
    "match_date": pick_col(matches, ["game_date"], required=False),
}

P = {
    "player_id": pick_col(players, ["player_id"]),
    "birth_date": pick_col(players, ["date_of_birth"]),
    "position": pick_col(players, ["position"], required=False),
}

print("PS:", PS)
print("M:", M)
print("P:", P)

PS: {'player_id': 'player_id', 'match_id': 'match_id', 'club_id': 'club_id', 'season': None, 'goals': 'goals', 'assists': 'assists', 'yellow_cards': 'yellow', 'red_cards': 'red', 'yellow_red_cards': 'yellow_red', 'started': 'start_eleven', 'minutes': 'minutes', 'rating': 'rating', 'team_goals': 'team_goals', 'team_conceded': 'team_conceded'}
M: {'match_id': 'match_id', 'season': 'season', 'competition': 'league', 'home_club_id': 'home_club_id', 'away_club_id': 'away_club_id', 'home_goals': 'home_goals', 'away_goals': 'away_goals', 'match_date': 'game_date'}
P: {'player_id': 'player_id', 'birth_date': 'date_of_birth', 'position': 'position'}


In [31]:
player_stats = player_stats.rename(columns={v: k for k, v in PS.items() if v is not None})
matches = matches.rename(columns={v: k for k, v in M.items() if v is not None})
players = players.rename(columns={v: k for k, v in P.items() if v is not None})

print(player_stats.columns.tolist())
print(matches.columns.tolist())
print(players.columns.tolist())

['player_id', 'match_id', 'club_id', 'goals', 'assists', 'yellow_cards', 'yellow_red_cards', 'red_cards', 'started', 'minutes', 'on_min', 'off_min', 'team_goals', 'team_conceded', 'rating']
['match_id', 'season', 'match_date', 'competition', 'home_club_id', 'away_club_id', 'home_goals', 'away_goals']
['player_id', 'player_name', 'nationality', 'birth_date', 'height', 'position']


In [32]:
def season_to_start_year(x):
    if pd.isna(x):
        return np.nan

    s = str(x).strip()

    # 20/21 oder 20-21 -> 2020
    m = re.fullmatch(r"(\d{2})\s*[/\-]\s*(\d{2})", s)
    if m:
        return 2000 + int(m.group(1))

    # 2020/21 oder 2020-21 -> 2020
    m = re.fullmatch(r"(20\d{2})\s*[/\-]\s*(\d{2})", s)
    if m:
        return int(m.group(1))

    # 2020/2021 oder 2020-2021 -> 2020
    m = re.fullmatch(r"(20\d{2})\s*[/\-]\s*(20\d{2})", s)
    if m:
        return int(m.group(1))

    # nur 2020
    m = re.fullmatch(r"20\d{2}", s)
    if m:
        return int(s)

    return np.nan


def age_on_aug1(season_start, birth_date):
    if pd.isna(season_start) or pd.isna(birth_date):
        return np.nan

    cutoff = pd.Timestamp(year=int(season_start), month=8, day=1)

    return cutoff.year - birth_date.year - (
        (cutoff.month, cutoff.day) < (birth_date.month, birth_date.day)
    )


def to_binary_started(x):
    if pd.isna(x):
        return 0

    if isinstance(x, (bool, np.bool_)):
        return int(x)

    if isinstance(x, (int, float, np.integer, np.floating)):
        return int(x > 0)

    s = str(x).strip().lower()
    if s in {"1", "true", "yes", "y"}:
        return 1
    return 0


def normalize_text(x):
    if pd.isna(x):
        return ""
    s = str(x).strip().lower()
    s = re.sub(r"\s+", " ", s)
    return s


def is_first_liga(comp):
    s = normalize_text(comp)
    return int(bool(re.search(r"\b1\.?\s*liga\b", s)))


def is_pl(comp):
    s = normalize_text(comp)
    return int(("promotion league" in s) or bool(re.search(r"\bpl\b", s)))


def get_result(row):
    if row["club_id"] == row["home_club_id"]:
        if row["home_goals"] > row["away_goals"]:
            return "win"
        elif row["home_goals"] < row["away_goals"]:
            return "loss"
        else:
            return "draw"

    elif row["club_id"] == row["away_club_id"]:
        if row["away_goals"] > row["home_goals"]:
            return "win"
        elif row["away_goals"] < row["home_goals"]:
            return "loss"
        else:
            return "draw"

    else:
        return None

In [33]:
for col in [
    "goals",
    "assists",
    "yellow_cards",
    "red_cards",
    "yellow_red_cards",
    "minutes",
    "rating",
    "team_goals",
    "team_conceded"
]:
    if col not in player_stats.columns:
        player_stats[col] = 0

for col in ["home_goals", "away_goals"]:
    if col not in matches.columns:
        matches[col] = np.nan

player_stats["rating"] = (
    player_stats["rating"]
    .astype(str)
    .str.replace(",", ".", regex=False)
)

num_cols_player_stats = [
    "goals",
    "assists",
    "yellow_cards",
    "red_cards",
    "yellow_red_cards",
    "minutes",
    "rating",
    "team_goals",
    "team_conceded"
]

for col in num_cols_player_stats:
    player_stats[col] = pd.to_numeric(player_stats[col], errors="coerce")

matches["home_goals"] = pd.to_numeric(matches["home_goals"], errors="coerce")
matches["away_goals"] = pd.to_numeric(matches["away_goals"], errors="coerce")

player_stats["started"] = player_stats["started"].apply(to_binary_started)
players["birth_date"] = pd.to_datetime(players["birth_date"], errors="coerce")

In [34]:
match_cols = ["match_id", "home_club_id", "away_club_id", "home_goals", "away_goals"]

if "season" in matches.columns:
    match_cols.append("season")
if "competition" in matches.columns:
    match_cols.append("competition")
if "match_date" in matches.columns:
    match_cols.append("match_date")

match_join = matches[match_cols].copy()

if "season" in match_join.columns:
    match_join = match_join.rename(columns={"season": "season_match"})

player_join = players[["player_id", "birth_date", "position"]].drop_duplicates(subset=["player_id"]).copy()

df = player_stats.merge(match_join, on="match_id", how="left")
df = df.merge(player_join, on="player_id", how="left")

# season aus player_stats, falls leer aus matches
if "season" not in df.columns:
    df["season"] = df["season_match"]
else:
    df["season"] = df["season"].fillna(df.get("season_match"))

df["season"] = df["season"].apply(season_to_start_year)

df = df.drop_duplicates(subset=["player_id", "match_id", "club_id"]).copy()

print("df shape:", df.shape)
display(df.head())

df shape: (146964, 25)


,player_id,match_id,club_id,goals,assists,yellow_cards,yellow_red_cards,red_cards,started,minutes,on_min,off_min,team_goals,team_conceded,rating,home_club_id,away_club_id,home_goals,away_goals,season_match,competition,match_date,birth_date,position,season
0,284695,3393584,322,0,0,NaN,NaN,NaN,0,90,0,0,2,0,7.5,322,8508,2,0,20/21,pl,2020-08-15,1995-06-13,Torwart,2020
1,115188,3393584,322,0,0,NaN,NaN,NaN,0,90,0,0,2,0,7.4,322,8508,2,0,20/21,pl,2020-08-15,1991-01-11,Innenverteidiger,2020
2,19279,3393584,322,0,0,NaN,NaN,NaN,0,90,0,0,2,0,7.4,322,8508,2,0,20/21,pl,2020-08-15,1986-01-03,Innenverteidiger,2020
3,126514,3393584,322,0,0,NaN,NaN,NaN,0,90,0,0,2,0,7.3,322,8508,2,0,20/21,pl,2020-08-15,1993-03-03,Linker Verteidiger,2020
4,267582,3393584,322,0,0,NaN,NaN,NaN,0,90,0,0,2,0,7.1,322,8508,2,0,20/21,pl,2020-08-15,1992-04-06,Rechter Verteidiger,2020


In [35]:
df["alter"] = df.apply(lambda row: age_on_aug1(row["season"], row["birth_date"]), axis=1)

df["result"] = df.apply(get_result, axis=1)

df["is_win"] = (df["result"] == "win").astype(int)
df["is_draw"] = (df["result"] == "draw").astype(int)
df["is_loss"] = (df["result"] == "loss").astype(int)

df["is_first_liga"] = df["competition"].apply(is_first_liga)
df["is_pl"] = df["competition"].apply(is_pl)

print(df["season"].value_counts(dropna=False).sort_index())
display(df.head())

season
2020    11656
2021    23339
2022    30037
2023    30895
2024    31208
2025    19829
Name: count, dtype: int64


,player_id,match_id,club_id,goals,assists,yellow_cards,yellow_red_cards,red_cards,started,minutes,on_min,off_min,team_goals,team_conceded,rating,home_club_id,away_club_id,home_goals,away_goals,season_match,competition,match_date,birth_date,position,season,alter,result,is_win,is_draw,is_loss,is_first_liga,is_pl
0,284695,3393584,322,0,0,NaN,NaN,NaN,0,90,0,0,2,0,7.5,322,8508,2,0,20/21,pl,2020-08-15,1995-06-13,Torwart,2020,25.0,win,1,0,0,0,1
1,115188,3393584,322,0,0,NaN,NaN,NaN,0,90,0,0,2,0,7.4,322,8508,2,0,20/21,pl,2020-08-15,1991-01-11,Innenverteidiger,2020,29.0,win,1,0,0,0,1
2,19279,3393584,322,0,0,NaN,NaN,NaN,0,90,0,0,2,0,7.4,322,8508,2,0,20/21,pl,2020-08-15,1986-01-03,Innenverteidiger,2020,34.0,win,1,0,0,0,1
3,126514,3393584,322,0,0,NaN,NaN,NaN,0,90,0,0,2,0,7.3,322,8508,2,0,20/21,pl,2020-08-15,1993-03-03,Linker Verteidiger,2020,27.0,win,1,0,0,0,1
4,267582,3393584,322,0,0,NaN,NaN,NaN,0,90,0,0,2,0,7.1,322,8508,2,0,20/21,pl,2020-08-15,1992-04-06,Rechter Verteidiger,2020,28.0,win,1,0,0,0,1


In [36]:
season_features = (
    df.groupby(["player_id", "season"], as_index=False)
      .agg(
          alter=("alter", "first"),
          position=("position", "first"),

          anzahl_spiele=("match_id", "nunique"),

          tore_abs=("goals", "sum"),
          assists_abs=("assists", "sum"),

          rote_karten_abs=("red_cards", "sum"),
          gelbe_karten_abs=("yellow_cards", "sum"),
          gelb_rote_karten_abs=("yellow_red_cards", "sum"),

          startelf_abs=("started", "sum"),
          minuten_abs=("minutes", "sum"),

          team_tore_abs=("team_goals", "sum"),
          team_gegentore_abs=("team_conceded", "sum"),

          siege_abs=("is_win", "sum"),
          unentschieden_abs=("is_draw", "sum"),
          niederlagen_abs=("is_loss", "sum"),

          prozent_1_liga=("is_first_liga", "mean"),
          prozent_pl=("is_pl", "mean"),

          avg_rating_current=("rating", "mean")
      )
)

season_features["prozent_1_liga"] = season_features["prozent_1_liga"] * 100
season_features["prozent_pl"] = season_features["prozent_pl"] * 100

season_features["tore_pro_spiel"] = season_features["tore_abs"] / season_features["anzahl_spiele"]
season_features["assists_pro_spiel"] = season_features["assists_abs"] / season_features["anzahl_spiele"]

season_features["rote_karten_pro_spiel"] = season_features["rote_karten_abs"] / season_features["anzahl_spiele"]
season_features["gelbe_karten_pro_spiel"] = season_features["gelbe_karten_abs"] / season_features["anzahl_spiele"]
season_features["gelb_rote_karten_pro_spiel"] = season_features["gelb_rote_karten_abs"] / season_features["anzahl_spiele"]

season_features["startelf_pro_spiel"] = season_features["startelf_abs"] / season_features["anzahl_spiele"]
season_features["minuten_pro_spiel"] = season_features["minuten_abs"] / season_features["anzahl_spiele"]

season_features["team_tore_pro_spiel"] = season_features["team_tore_abs"] / season_features["anzahl_spiele"]
season_features["team_gegentore_pro_spiel"] = season_features["team_gegentore_abs"] / season_features["anzahl_spiele"]

season_features["siege_pro_spiel"] = season_features["siege_abs"] / season_features["anzahl_spiele"]
season_features["unentschieden_pro_spiel"] = season_features["unentschieden_abs"] / season_features["anzahl_spiele"]
season_features["niederlagen_pro_spiel"] = season_features["niederlagen_abs"] / season_features["anzahl_spiele"]

print("season_features shape:", season_features.shape)
display(season_features.head())

season_features shape: (11198, 32)


,player_id,season,alter,position,anzahl_spiele,tore_abs,assists_abs,rote_karten_abs,gelbe_karten_abs,gelb_rote_karten_abs,startelf_abs,minuten_abs,team_tore_abs,team_gegentore_abs,siege_abs,unentschieden_abs,niederlagen_abs,prozent_1_liga,prozent_pl,avg_rating_current,tore_pro_spiel,assists_pro_spiel,rote_karten_pro_spiel,gelbe_karten_pro_spiel,gelb_rote_karten_pro_spiel,startelf_pro_spiel,minuten_pro_spiel,team_tore_pro_spiel,team_gegentore_pro_spiel,siege_pro_spiel,unentschieden_pro_spiel,niederlagen_pro_spiel
0,2452,2020,36.0,Mittelstürmer,6,1,0,0.0,0.0,0.0,0,229,3,6,2,3,1,0.0,33.333333,6.683333,0.166667,0.0,0.0,0.0,0.0,0.0,38.166667,0.500000,1.000000,0.333333,0.500000,0.166667
1,2866,2020,37.0,Innenverteidiger,10,0,0,0.0,0.0,0.0,0,806,21,17,6,2,2,0.0,0.000000,6.800000,0.000000,0.0,0.0,0.0,0.0,0.0,80.600000,2.100000,1.700000,0.600000,0.200000,0.200000
2,2866,2021,38.0,Innenverteidiger,19,2,0,0.0,0.0,0.0,0,1090,26,14,12,1,6,0.0,0.000000,6.989474,0.105263,0.0,0.0,0.0,0.0,0.0,57.368421,1.368421,0.736842,0.631579,0.052632,0.315789
3,2866,2024,41.0,Innenverteidiger,8,0,0,0.0,0.0,0.0,0,284,9,7,6,0,2,0.0,0.000000,6.725000,0.000000,0.0,0.0,0.0,0.0,0.0,35.500000,1.125000,0.875000,0.750000,0.000000,0.250000
4,3391,2020,34.0,Offensives Mittelfeld,3,1,0,0.0,0.0,0.0,0,209,6,3,3,0,0,0.0,0.000000,7.233333,0.333333,0.0,0.0,0.0,0.0,0.0,69.666667,2.000000,1.000000,1.000000,0.000000,0.000000


In [37]:
tmp = season_features.copy()
tmp = tmp.sort_values(["player_id", "season"]).reset_index(drop=True)

tmp["next_season"] = tmp.groupby("player_id")["season"].shift(-1)
tmp["ziel_rating_avg_naechste_saison"] = tmp.groupby("player_id")["avg_rating_current"].shift(-1)

tmp = tmp[tmp["next_season"] == tmp["season"] + 1].copy()
tmp = tmp[(tmp["season"] >= 2020) & (tmp["season"] <= 2024)].copy()

model_df = tmp.dropna(subset=["ziel_rating_avg_naechste_saison"]).reset_index(drop=True)

print("model_df shape:", model_df.shape)
print(model_df["season"].value_counts().sort_index())
display(model_df.head())

model_df shape: (5643, 34)
season
2020     987
2021    1047
2022    1214
2023    1245
2024    1150
Name: count, dtype: int64


,player_id,season,alter,position,anzahl_spiele,tore_abs,assists_abs,rote_karten_abs,gelbe_karten_abs,gelb_rote_karten_abs,startelf_abs,minuten_abs,team_tore_abs,team_gegentore_abs,siege_abs,unentschieden_abs,niederlagen_abs,prozent_1_liga,prozent_pl,avg_rating_current,tore_pro_spiel,assists_pro_spiel,rote_karten_pro_spiel,gelbe_karten_pro_spiel,gelb_rote_karten_pro_spiel,startelf_pro_spiel,minuten_pro_spiel,team_tore_pro_spiel,team_gegentore_pro_spiel,siege_pro_spiel,unentschieden_pro_spiel,niederlagen_pro_spiel,next_season,ziel_rating_avg_naechste_saison
0,2866,2020,37.0,Innenverteidiger,10,0,0,0.0,0.0,0.0,0,806,21,17,6,2,2,0.0,0.0,6.800000,0.000000,0.000000,0.0,0.0,0.0,0.0,80.600000,2.100000,1.700000,0.600000,0.200000,0.200000,2021.0,6.989474
1,3391,2020,34.0,Offensives Mittelfeld,3,1,0,0.0,0.0,0.0,0,209,6,3,3,0,0,0.0,0.0,7.233333,0.333333,0.000000,0.0,0.0,0.0,0.0,69.666667,2.000000,1.000000,1.000000,0.000000,0.000000,2021.0,6.900000
2,4779,2021,39.0,Offensives Mittelfeld,24,2,1,0.0,0.0,0.0,0,1884,35,46,8,4,12,0.0,0.0,6.850000,0.083333,0.041667,0.0,0.0,0.0,0.0,78.500000,1.458333,1.916667,0.333333,0.166667,0.500000,2022.0,6.720000
3,10058,2020,35.0,Innenverteidiger,12,0,0,0.0,0.0,0.0,0,1003,10,13,6,2,4,0.0,100.0,7.041667,0.000000,0.000000,0.0,0.0,0.0,0.0,83.583333,0.833333,1.083333,0.500000,0.166667,0.333333,2021.0,6.953333
4,10058,2021,36.0,Innenverteidiger,30,0,0,0.0,0.0,0.0,0,2700,44,41,12,7,11,0.0,100.0,6.953333,0.000000,0.000000,0.0,0.0,0.0,0.0,90.000000,1.466667,1.366667,0.400000,0.233333,0.366667,2022.0,6.737931


In [38]:
final_columns = [
    "player_id",
    "season",
    "alter",
    "position",

    "tore_pro_spiel",
    "tore_abs",

    "assists_pro_spiel",
    "assists_abs",

    "rote_karten_pro_spiel",
    "gelbe_karten_pro_spiel",
    "gelb_rote_karten_pro_spiel",

    "rote_karten_abs",
    "gelbe_karten_abs",
    "gelb_rote_karten_abs",

    "startelf_pro_spiel",
    "startelf_abs",

    "minuten_pro_spiel",
    "minuten_abs",

    "anzahl_spiele",

    "team_tore_pro_spiel",
    "team_tore_abs",

    "team_gegentore_pro_spiel",
    "team_gegentore_abs",

    "siege_pro_spiel",
    "unentschieden_pro_spiel",
    "niederlagen_pro_spiel",

    "siege_abs",
    "unentschieden_abs",
    "niederlagen_abs",

    "prozent_1_liga",
    "prozent_pl",

    "ziel_rating_avg_naechste_saison"
]

final_dataset = model_df[final_columns].copy()

print(final_dataset.shape)
display(final_dataset.head())

(5643, 32)


,player_id,season,alter,position,tore_pro_spiel,tore_abs,assists_pro_spiel,assists_abs,rote_karten_pro_spiel,gelbe_karten_pro_spiel,gelb_rote_karten_pro_spiel,rote_karten_abs,gelbe_karten_abs,gelb_rote_karten_abs,startelf_pro_spiel,startelf_abs,minuten_pro_spiel,minuten_abs,anzahl_spiele,team_tore_pro_spiel,team_tore_abs,team_gegentore_pro_spiel,team_gegentore_abs,siege_pro_spiel,unentschieden_pro_spiel,niederlagen_pro_spiel,siege_abs,unentschieden_abs,niederlagen_abs,prozent_1_liga,prozent_pl,ziel_rating_avg_naechste_saison
0,2866,2020,37.0,Innenverteidiger,0.000000,0,0.000000,0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0,80.600000,806,10,2.100000,21,1.700000,17,0.600000,0.200000,0.200000,6,2,2,0.0,0.0,6.989474
1,3391,2020,34.0,Offensives Mittelfeld,0.333333,1,0.000000,0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0,69.666667,209,3,2.000000,6,1.000000,3,1.000000,0.000000,0.000000,3,0,0,0.0,0.0,6.900000
2,4779,2021,39.0,Offensives Mittelfeld,0.083333,2,0.041667,1,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0,78.500000,1884,24,1.458333,35,1.916667,46,0.333333,0.166667,0.500000,8,4,12,0.0,0.0,6.720000
3,10058,2020,35.0,Innenverteidiger,0.000000,0,0.000000,0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0,83.583333,1003,12,0.833333,10,1.083333,13,0.500000,0.166667,0.333333,6,2,4,0.0,100.0,6.953333
4,10058,2021,36.0,Innenverteidiger,0.000000,0,0.000000,0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0,90.000000,2700,30,1.466667,44,1.366667,41,0.400000,0.233333,0.366667,12,7,11,0.0,100.0,6.737931


In [39]:
final_dataset["position"] = final_dataset["position"].fillna("unknown").astype(str).str.strip()

final_dataset = pd.get_dummies(
    final_dataset,
    columns=["position"],
    prefix="position",
    dtype=int
)

print(final_dataset.shape)
display(final_dataset.head())

(5643, 47)


,player_id,season,alter,tore_pro_spiel,tore_abs,assists_pro_spiel,assists_abs,rote_karten_pro_spiel,gelbe_karten_pro_spiel,gelb_rote_karten_pro_spiel,rote_karten_abs,gelbe_karten_abs,gelb_rote_karten_abs,startelf_pro_spiel,startelf_abs,minuten_pro_spiel,minuten_abs,anzahl_spiele,team_tore_pro_spiel,team_tore_abs,team_gegentore_pro_spiel,team_gegentore_abs,siege_pro_spiel,unentschieden_pro_spiel,niederlagen_pro_spiel,siege_abs,unentschieden_abs,niederlagen_abs,prozent_1_liga,prozent_pl,ziel_rating_avg_naechste_saison,position_Abwehr,position_Defensives Mittelfeld,position_Hängende Spitze,position_Innenverteidiger,position_Linker Verteidiger,position_Linkes Mittelfeld,position_Linksaußen,position_Mittelfeld,position_Mittelstürmer,position_Offensives Mittelfeld,position_Rechter Verteidiger,position_Rechtes Mittelfeld,position_Rechtsaußen,position_Sturm,position_Torwart,position_Zentrales Mittelfeld
0,2866,2020,37.0,0.000000,0,0.000000,0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0,80.600000,806,10,2.100000,21,1.700000,17,0.600000,0.200000,0.200000,6,2,2,0.0,0.0,6.989474,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0
1,3391,2020,34.0,0.333333,1,0.000000,0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0,69.666667,209,3,2.000000,6,1.000000,3,1.000000,0.000000,0.000000,3,0,0,0.0,0.0,6.900000,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0
2,4779,2021,39.0,0.083333,2,0.041667,1,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0,78.500000,1884,24,1.458333,35,1.916667,46,0.333333,0.166667,0.500000,8,4,12,0.0,0.0,6.720000,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0
3,10058,2020,35.0,0.000000,0,0.000000,0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0,83.583333,1003,12,0.833333,10,1.083333,13,0.500000,0.166667,0.333333,6,2,4,0.0,100.0,6.953333,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0
4,10058,2021,36.0,0.000000,0,0.000000,0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0,90.000000,2700,30,1.466667,44,1.366667,41,0.400000,0.233333,0.366667,12,7,11,0.0,100.0,6.737931,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0


In [40]:
missing = final_dataset.isna().mean().sort_values(ascending=False)
display(missing[missing > 0])

alter    0.000709
dtype: float64

In [41]:
print("Saisons im finalen Datensatz:")
print(final_dataset["season"].value_counts().sort_index())

print("\nAnzahl Spieler:")
print(final_dataset["player_id"].nunique())

print("\nTarget Summary:")
print(final_dataset["ziel_rating_avg_naechste_saison"].describe())

Saisons im finalen Datensatz:
season
2020     987
2021    1047
2022    1214
2023    1245
2024    1150
Name: count, dtype: int64

Anzahl Spieler:
2568

Target Summary:
count    5643.000000
mean        6.846437
std         0.229117
min         3.700000
25%         6.700000
50%         6.832143
75%         6.976000
max         8.400000
Name: ziel_rating_avg_naechste_saison, dtype: float64


In [42]:
position_cols = [c for c in final_dataset.columns if c.startswith("position_")]
print(position_cols)

['position_Abwehr', 'position_Defensives Mittelfeld', 'position_Hängende Spitze', 'position_Innenverteidiger', 'position_Linker Verteidiger', 'position_Linkes Mittelfeld', 'position_Linksaußen', 'position_Mittelfeld', 'position_Mittelstürmer', 'position_Offensives Mittelfeld', 'position_Rechter Verteidiger', 'position_Rechtes Mittelfeld', 'position_Rechtsaußen', 'position_Sturm', 'position_Torwart', 'position_Zentrales Mittelfeld']


In [44]:
final_dataset.to_csv("recommender_model_dataset.csv", index=False)
print("Saved as: recommender_model_dataset.csv")

Saved as: recommender_model_dataset.csv
